# Módulo AI5 · Fine-tuning supervisado — Práctica

SFT juguete en CPU con PyTorch: dataset de pares (instrucción, respuesta) → chat template → máscara de pérdida → entrenamiento → comparar con/ sin máscara. Luego LoRA a mano sobre un linear, y el esqueleto TRL/PEFT para Colab.


In [ ]:
import json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42); np.random.seed(42)

In [ ]:
# Dataset SFT: 16 pares de intenciones de TiendaSur
DATOS = [
    ("¿cuánto tarda el envío?", "El envío estándar tarda 3-5 días laborables."),
    ("¿y si necesito el pedido mañana?", "Pide envío urgente antes de las 14:00 y llega al día siguiente."),
    ("¿cuándo llega mi factura?", "La factura se emite el mismo día de la compra y llega por correo."),
    ("puedo pedir factura con CIF?", "Sí: se emite en 24 h con datos fiscales completos."),
    ("¿hasta cuándo devuelvo?", "Tienes 30 días naturales, con el producto sin usar."),
    ("quién paga la devolución", "El cliente la paga, salvo producto defectuoso."),
    ("¿cuánto tarda la factura con cif?", "Se emite en 24 horas con tus datos fiscales completos."),
    ("¿quién paga si el producto llega defectuoso?", "Nosotros: el envío de devolución corre a nuestro cargo."),
    ("¿cuándo llega el pedido urgente?", "Al día siguiente, si lo pides antes de las 14:00."),
    ("puedo devolver el producto usado?", "No: los 30 días valen con el producto sin usar."),
    ("¿cuánto tarda el envío a la península?", "El estándar: 3-5 días laborables."),
    ("¿la factura con cif tarda?", "Se emite en 24 h con datos fiscales completos."),
    ("¿la devolución de un producto defectuoso?", "El envío lo pagamos nosotros."),
    ("¿cuándo llega la factura con cif?", "En 24 horas, con tus datos fiscales completos."),
    ("¿cómo es el envío urgente?", "Llega al día siguiente si se pide antes de las 14:00."),
    ("¿puedo devolver usado?", "No: debe estar sin usar, dentro de los 30 días."),
]

## 2 · Chat template + máscara de pérdida

La clave del SFT: **solo se aprende de la respuesta** (`<resp>...`), no del prompt. Enmascarar la pérdida = poner -100 en las posiciones de prompt.

In [ ]:
import collections
tokens = sorted(set(c for s, r in DATOS for c in s + r))
tok2id = {t: i for i, t in enumerate(tokens)}
SPECIAL = {"<instr>": len(tok2id), "<resp>": len(tok2id)+1, "<fin>": len(tok2id)+2}
tok2id.update(SPECIAL); id2tok = {i: t for t, i in tok2id.items()}
V = len(tok2id)
print("vocab:", V)

def encode_par(instruccion, respuesta, ctx=48):
    """Convierte (instrucción, respuesta) a (input, labels) desplazados;
    labels = -100 en el prompt: el modelo solo aprende de la respuesta."""
    seq = [SPECIAL['<instr>']] + [tok2id[c] for c in instruccion] \
        + [SPECIAL['<resp>']] + [tok2id[c] for c in respuesta] + [SPECIAL['<fin>']]
    seq = seq[:ctx]
    inp = seq[:-1]; tgt = seq[1:]
    labels = [t if t != SPECIAL['<resp>'] else -100 for t in tgt]
    inp = inp + [0]*(ctx-1-len(inp)); labels = labels + [-100]*(ctx-1-len(labels))
    return torch.tensor([inp]), torch.tensor([labels])

xb, yb = encode_par(*DATOS[0])
print("input :", "".join(id2tok[i] for i in xb[0].tolist() if i in id2tok)[:80], "…")
print("labels:", yb[0].tolist()[:14])

## 3 · Modelo SFT juguete (misma arquitectura del AI1, más pequeña)

In [ ]:
class Bloque(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.qkv = nn.Linear(dim, 3*dim); self.proj = nn.Linear(dim, dim)
        self.mlp = nn.Sequential(nn.Linear(dim, 4*dim), nn.GELU(), nn.Linear(4*dim, dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.h = heads
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = (t.view(B, T, self.h, -1).transpose(1, 2) for t in (q, k, v))
        a = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
        a = a.masked_fill(torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), 1), float('-inf'))
        a = F.softmax(a, -1) @ v
        a = a.transpose(1, 2).reshape(B, T, C)
        x = x + self.proj(self.ln1(a))
        return x + self.mlp(self.ln2(x))

class SFTModel(nn.Module):
    def __init__(self, vocab, dim=64, layers=2, ctx=48):
        super().__init__()
        self.ctx = ctx
        self.tok = nn.Embedding(vocab, dim)
        self.pos = nn.Embedding(ctx, dim)
        self.blocks = nn.Sequential(*[Bloque(dim, 4) for _ in range(layers)])
        self.ln = nn.LayerNorm(dim); self.head = nn.Linear(dim, vocab)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.blocks(x))

## 4 · Entrenar SFT **con máscara** vs **sin máscara**

In [ ]:
def sft_train(usar_mascara=True, pasos=400, lr=1e-3):
    m = SFTModel(V); opt = torch.optim.AdamW(m.parameters(), lr=lr)
    for paso in range(pasos):
        i = np.random.randint(0, len(DATOS))
        xb, labels = encode_par(*DATOS[i])
        if not usar_mascara:
            # sin máscara: aprende de todo el texto (prompt también)
            seq_ids = xb[0].clone()
            labels = seq_ids[1:]; xb_in = seq_ids[:-1].unsqueeze(0)
            # reconstruir el seq completo para el "target" sin máscara:
            full = torch.cat([xb[0], torch.tensor([SPECIAL['<fin>']])])
            tgt_sin_mask = full[1:]
            inp_sin = full[:-1].unsqueeze(0)
            logits = m(inp_sin)
            loss = F.cross_entropy(logits.reshape(-1, V), tgt_sin_mask)
        else:
            logits = m(xb)
            loss = F.cross_entropy(logits.reshape(-1, V), labels.reshape(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
    return m, loss.item()

m_mask, l_mask = sft_train(usar_mascara=True)
m_nomask, l_nom = sft_train(usar_mascara=False)
print(f"loss con máscara (solo respuesta): {l_mask:.3f}")
print(f"loss sin máscara (todo el texto):  {l_nom:.3f}")

## 5 · Generar respuestas con el modelo SFT juguete

In [ ]:
@torch.no_grad()
def generar_sft(m, instruccion, max_n=20, temperature=0.7):
    ids = [SPECIAL['<instr>']] + [tok2id[c] for c in instruccion] + [SPECIAL['<resp>']]
    for _ in range(max_n):
        ctx_t = torch.tensor([ids[-m.ctx:]], dtype=torch.long)
        logits = m(ctx_t)[0, -1]
        probs = F.softmax(logits / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1).item()
        if nxt == SPECIAL['<fin>']:
            break
        ids.append(nxt if nxt in id2tok else SPECIAL['<fin>'])
    return "".join(id2tok.get(i, "") for i in ids).split("<resp>")[-1]

for q in ["¿cuánto tarda el envío?", "¿quién paga la devolución?", "¿cuándo llega la factura con CIF?"]:
    print(f"Q: {q}")
    print(f"  con máscara → {generar_sft(m_mask, q)!r}")
    print(f"  sin máscara → {generar_sft(m_nomask, q)!r}")

## 6 · LoRA a mano: solo entrenar el adaptador

LoRA: `W' = W + (A @ B)` con A (r×in) y B (out×r). Congelamos W y entrenamos A,B.


In [ ]:
# LoRA a mano sobre la proyección qkv del bloque 0.
# W' = W + ΔW, con ΔW = B @ A (LoRA paper): A (r, in), B (out, r)
W = m_mask.blocks[0].qkv.weight.detach().clone()   # (out=192, in=64)
r = 4
A = nn.Parameter(torch.randn(r, W.shape[1]) * 0.02)  # (r, in)
B = nn.Parameter(torch.zeros(W.shape[0], r))         # (out, r): arranca en 0 → W' = W al inicio

def sft_lora(pasos=400, lr=1e-3):
    opt = torch.optim.AdamW([A, B], lr=lr)
    for _ in range(pasos):
        i = np.random.randint(0, len(DATOS))
        xb, labels = encode_par(*PAR[i]) if False else encode_par(*DATOS[i])
        with torch.no_grad():
            e = m_mask.tok(xb) + m_mask.pos(torch.arange(xb.shape[1]))
            x = m_mask.blocks[0].ln1(e)
        B_, T, C = x.shape
        qkv = x @ (W + B @ A).T          # (B,T,in) @ (in,out) → (B,T,out)
        q, k, v = qkv.split(C, dim=2)
        q, k, v = (t.view(B_, T, 4, -1).transpose(1, 2) for t in (q, k, v))
        a = (q @ k.transpose(-2, -1)) / math.sqrt(q.size(-1))
        a = a.masked_fill(torch.triu(torch.ones(T, T, dtype=torch.bool), 1), float("-inf"))
        a = (F.softmax(a, -1) @ v).transpose(1, 2).reshape(B_, T, C)
        logits = m_mask.head(m_mask.ln(m_mask.blocks[0].proj(a)))
        loss = F.cross_entropy(logits.reshape(-1, V), labels.reshape(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
    return float(loss)

l_lora = sft_lora()
print(f"LoRA: entrenados {A.numel()+B.numel()} parámetros de {W.size(0)*W.size(1)} congelados")
print(f"loss LoRA tras 400 pasos: {l_lora:.3f}")

## 7 · [Colab] Esqueleto TRL + PEFT para modelo real

Esto requiere GPU (Colab T4) e internet. Lo dejas preparado; lo ejecutas desde Colab:

```python
# %pip install -q transformers trl peft accelerate datasets
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer
from peft import LoraConfig

base = "Qwen/Qwen2.5-0.5B-Instruct"   # u otro abierto pequeño (Gemma/SmolLM)
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base)
cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"])
# dataset = tus pares {"instruccion": ..., "respuesta": ...} con chat template del base
# trainer = SFTTrainer(model=..., train_dataset=..., peft_config=cfg, ...)
# trainer.train(); trainer.save_layout("sft-tiendasur")
```


---

## Ejercicios (5)
1. Duplica `DATOS` (32 pares). ¿Baja el loss? ¿Memoriza más o generaliza?
2. Entrena con más pasos (600). ¿El loss con máscara cae por debajo de 0.05? ¿Qué pasa con la generación (¿repite exacto)?  → overfitting.
3. Rompe el template: quita el token `<resp>`/`<fin>`. ¿Por qué la máscara deja de funcionar? (ver las labels).
4. Haz LoRA con r=16. ¿Más parámetros entrenan más rápido o mejor loss? ¿Cuándo **no** vale la pena?
5. *(Colab)* Ejecuta el esqueleto TRL con tu dataset de 10 pares y genera. En una frase: ¿qué mejora el SFT real respecto a tu juguete? (mundo del vocab, base preentrenada de datos masivos…)
